## This notebook runs the following tests.

### Test 1: Testing Data fit (lam_data = 1, all other lambda's set to 0)

In [3]:
import numpy as np
import sys

sys.path.append('..')  # add project root

import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import Datasets.load_mat_as_flat as lmf
import os

from prog import mlps, featlib, trainer, hlprs
import Datasets.matconv as mc

import importlib
importlib.reload(mlps)
importlib.reload(trainer)
importlib.reload(hlprs)
importlib.reload(lmf)

<module 'Datasets.load_mat_as_flat' from 'c:\\Users\\Ethan\\Desktop\\Research\\pde_stuff\\ethan_work\\..\\Datasets\\load_mat_as_flat.py'>

In [4]:
mat_path = "../Datasets/data/Diff_Curvature_ghost_pts/Allen_Cahn_tanh_2.0.mat"

lam_pde = 0.0
lam_tv = 0.0
lr_tv = 1e-4
lr_pde = 1e-3

lam_reg = 0.0
lam_data = 1.0

# hyperparameters for data generation
# data = sio.loadmat("path")
noise=0.0
nu=0.02
stride_t=1
stride_x=1
part_num=1
which_part=1
seed=1432

selected_derivs = ('u','u_x','u_xx')

# correct_coeffs = [0.0, 0.0, nu, -1.0]  # corresponds to the correct PDE: u_t + u*u_x - nu*u_xx = 0
correct_coeffs = [5, 0, 0.001, 0, -5]# u_t = 0.001u_xx + 5u - 5u^3


lr = 1e-3
batch_size = 1000
# epoch = data["t"].shape[0] * data["x"].shape[1]
# steps_per_epoch = epoch // batch_size
steps = 6000
log_every = 500

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
def make_models():
    u_model = mlps.SirenMLP(hidden_layers=4, hidden_size=64, first_omega_0=1.0, hidden_omega_0=1.0)
    symnet = mlps.EQL(in_dim=len(selected_derivs), prod_dim=2, num_layers=2, bias=False) # we need 2 linear layers to create the cubic terms
    return u_model, symnet

def make_trainer(u_model, symnet):
    train_config = trainer.TrainerConfig(
        lr=lr,
        lr_tv=lr_tv,
        lr_pde=lr_pde,
        lambda_pde= lam_pde,
        lambda_reg=lam_reg,
        lambda_tv= lam_tv,
        lambda_data=lam_data,
        lambda_tv_mask_fn = None,
        lambda_pde_mask_fn = None,
        selected_derivs=selected_derivs,
        device=device,
    )

    ft = featlib.FeatureTensor(selected_derivs, normalize=False)
    feature_builder = ft.build

    return trainer.PDETrainer(
        u_model=u_model,
        v_model=symnet,
        cfg=train_config,
        feature_builder=feature_builder,
    )

def train_model(train, steps, t_torch, x_torch, u_noisy, u_clean, y_clean_torch, y_noisy_torch):
    out = []
    for i in range(steps): 
        t,x,u_noisy,u_clean = hlprs.make_batch(batch_size=batch_size,
                                            t_torch=t_torch,
                                            x_torch=x_torch,
                                            y_clean=y_clean_torch,
                                            y_noisy=y_noisy_torch)
        out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
        if i % 300 == 0:
            print(f"Step {i}: loss: {out['loss']}")
    return out

In [6]:
partitions = lmf.load_burgers_mat_as_flat(mat_path=mat_path,
                                                noise_level=noise, 
                                                stride_t=stride_t, 
                                                stride_x=stride_x, 
                                                seed=seed, 
                                                quantile_splits=part_num, 
                                                return_partitions=True)

key = [k for k in partitions if k.startswith(f'Q{which_part}:')][0]
t_np, x_np, y_np, y_noisy_np, N = partitions[key]
# torch tensors
t_torch       = torch.from_numpy(t_np).to(device)
x_torch       = torch.from_numpy(x_np).to(device)
y_clean_torch = torch.from_numpy(y_np).to(device)
y_noisy_torch = torch.from_numpy(y_noisy_np).to(device)

In [21]:
u_model, symnet = make_models()
train = make_trainer(u_model, symnet)

#train model 
out = []
for i in range(6000): 
    t,x,u_noisy,u_clean = hlprs.make_batch(batch_size=batch_size,
                                        t_torch=t_torch,
                                        x_torch=x_torch,
                                        y_clean=y_clean_torch,
                                        y_noisy=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    if i % 300 == 0:
        print(f"Step {i}: loss: {out['loss']}")

hlprs.snapshot_comp(train.u, 
                    stride_x, 
                    stride_t, 
                    y_noisy_np, 
                    y_np, 
                    t_np, 
                    x_np, 
                    snap_no=10, 
                    folder=f"../ethan_work",
                    lam_tv=lam_tv
)[:0]

# save model instance to JSON
hlprs.save_to_json(train, out, correct_coeffs, path=f"../ethan_work/eps_2.0_data_fit.json")

Step 0: loss: 1.374255657196045
Step 300: loss: 0.006965659558773041
Step 600: loss: 0.0030615755822509527
Step 900: loss: 0.0017303221393376589
Step 1200: loss: 0.0009941067546606064
Step 1500: loss: 0.0005126585019752383
Step 1800: loss: 0.0004258067056071013
Step 2100: loss: 0.00034126118407584727
Step 2400: loss: 0.00017911079339683056
Step 2700: loss: 0.00019859953317791224
Step 3000: loss: 0.00013660681725014
Step 3300: loss: 0.000312066258629784
Step 3600: loss: 0.0001593335036886856
Step 3900: loss: 0.0001419476611772552
Step 4200: loss: 7.923947850940749e-05
Step 4500: loss: 0.0001603008568054065
Step 4800: loss: 4.301305307308212e-05
Step 5100: loss: 9.072804095922038e-05
Step 5400: loss: 0.00010852739069378003
Step 5700: loss: 3.3614676794968545e-05
max_abs_diff = 0.0862123


In [25]:
lam_data = 10.0
lam_pde = 0.5

u_model, symnet = make_models()
train = make_trainer(u_model, symnet)

out = []
for i in range(20000): 
    t,x,u_noisy,u_clean = hlprs.make_batch(batch_size=batch_size,
                                        t_torch=t_torch,
                                        x_torch=x_torch,
                                        y_clean=y_clean_torch,
                                        y_noisy=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    if i % 300 == 0:
        print(f"Step {i}: loss: {out['loss']}")

hlprs.snapshot_comp(train.u, 
                    stride_x, 
                    stride_t, 
                    y_noisy_np, 
                    y_np, 
                    t_np, 
                    x_np, 
                    snap_no=10, 
                    folder=f"../ethan_work",
                    lam_tv=lam_tv
)[:0]

# save model instance to JSON
hlprs.save_to_json(train, out, correct_coeffs, path=f"../ethan_work/eps_2.0_pde_fit.json")

Step 0: loss: 5.483379364013672
Step 300: loss: 0.6827682256698608
Step 600: loss: 0.29397648572921753
Step 900: loss: 0.2009139358997345
Step 1200: loss: 0.17113906145095825
Step 1500: loss: 0.13825394213199615
Step 1800: loss: 0.07110070437192917
Step 2100: loss: 0.04575377702713013
Step 2400: loss: 0.039639558643102646
Step 2700: loss: 0.043429289013147354
Step 3000: loss: 0.02895301580429077
Step 3300: loss: 0.02919614687561989
Step 3600: loss: 0.02506004273891449
Step 3900: loss: 0.027084829285740852
Step 4200: loss: 0.024081289768218994
Step 4500: loss: 0.021788178011775017
Step 4800: loss: 0.019439242780208588
Step 5100: loss: 0.02032901719212532
Step 5400: loss: 0.018629146739840508
Step 5700: loss: 0.018768755719065666
Step 6000: loss: 0.021849192678928375
Step 6300: loss: 0.02135636657476425
Step 6600: loss: 0.020270435139536858
Step 6900: loss: 0.018264289945364
Step 7200: loss: 0.017189154401421547
Step 7500: loss: 0.01999574340879917
Step 7800: loss: 0.01682838425040245
St

### Since we aren't recovering the PDE, we should try some debugging.
- Plug in correct coefficients, take one step and evaluate loss (should be very small)
- Compute derivatives through backprop and finite difference, for each derivative compare the two types of calculations on a graph
(note: we'll need to pre-train data fitting before we calculate)

In [10]:
u_model, symnet = make_models()
train = make_trainer(u_model, symnet)
with torch.no_grad():
    symnet.linears[0].weight.copy_(torch.tensor([[1, 0, 0,],
                                                [1, 0, 0,]],
                                                dtype=symnet.linears[0].weight.dtype, 
                                                device=symnet.linears[0].weight.device))
    symnet.linears[1].weight.copy_(torch.tensor([[1, 0, 0, 0],
                                                [0, 0, 0, 1]],
                                                dtype=symnet.linears[1].weight.dtype,
                                                device=symnet.linears[1].weight.device))
    symnet.readout.weight.copy_(torch.tensor([[5, 0, 0.001, 0, -5]],
                                             dtype=symnet.readout.weight.dtype,
                                             device=symnet.readout.weight.device))

In [11]:
symnet_error = train.mse(train.u_t,symnet(train.F))

AttributeError: 'PDETrainer' object has no attribute 'u_t'